In [1]:
import os 
import numpy as np 
import pickle
import datetime as dt 

---

**Creating a stability dataset of the simulations**  
Label corresponds to whether a layout is statically stable. 

In [2]:
NUM_SAMPLES_PER_TRAJECTORY = 5  # all samples of some trajectory T have the same label 
SPLIT = 0.7

In [3]:
num_simulations = len([filename for filename in os.listdir("cuboid_simulations") if ".npy" in filename])
num_train_simulations = int(SPLIT*num_simulations)
print(num_simulations, "total simulations")
print(num_train_simulations, "training simulations")

1000 total simulations
700 training simulations


In [4]:
def assess_stability(v):
    # TODO: Figure out how to assess the stability based on the trajectories of all items in a simulation. 
    #       If it boils down to comparing the beginning with the end, I need to consider that many items start in the air.
    #       If this is a problem, I might only be able to create this stability dataset once I implemented a way to create
    #       realistic layouts rather than just random ones.
    #       But in either case, I might have to incorporate some kind of "relaxation" phase or to incorporate some kind of 
    #       "relaxation" in the position delta. 
    #
    #
    #
    #
    #
    #
    #
    return np.random.choice([0, 1])

In [9]:
V = []
Y = []
for filename in os.listdir("cuboid_simulations"):
    if ".npy" in filename:
        v = np.load(open("cuboid_simulations/" + filename, "rb"))
        # 500+ samples for one item's trajectory in one particular simulation is probably too dense
        # I will sample NUM_SAMPLES_PER_TRAJECTORY entries, and try to have enough simulations and items 
        # to have a large and diverse dataset
        stability_score = assess_stability(v)
        y = [stability_score] * NUM_SAMPLES_PER_TRAJECTORY
        Y.extend(y)
        v = v[np.random.choice(range(len(v)), size=min(NUM_SAMPLES_PER_TRAJECTORY, len(v)), replace=False)]
        # Here I don't care about different timesteps. I simply want a list of 11-vectors
        # TODO: be able to deal with examples of varying number of items and timesteps
        #       e.g. by filling them up with "null items"
        #       but what would be even better (though I'm not sure how to do this yet), is to actually have them be empty
        #       and then later let the model learn an embedding corresponding to an [EMPTY] token
        #
        #
        #
        #
        V.append(v)
        
Y = np.array(Y)

V_train = np.concatenate(V[:num_train_simulations])
V_test = np.concatenate(V[num_train_simulations:]) 

# Normalize the data using statistics from the training set
mean, std = V_train.mean(), V_train.std()
V_train = (V_train-mean)/std
V_test = (V_test-mean)/std

# No longer needed
del V, mean, std

In [7]:
# """
# Store the whole dataset (before applying embeddings since I'm just using a random linear projection here for experimentation) 
# as a single pickled file

timestamp = str(dt.datetime.now()).replace(" ", "_").replace(":", "_").replace("-", "_").replace(".", "_")
pickle.dump((V_train, V_test), open(f"datasets/stability/dataset_{timestamp}.pickle", "wb"))

# E.g. in Colab, read it like this:
# V_train, V_test = pickle.load(open("datasets/stability/dataset_2023_01_18_17_06_06_959337.pickle", "rb"))
# """;

In [10]:
_V_train, _V_test = pickle.load(open("datasets/stability/dataset_2023_01_18_20_28_55_599444.pickle", "rb"))

In [14]:
_V_test.shape

(1500, 10, 11)